# Setup

In [1]:
# Install (run once at the top of the notebook, in its own cell)
!pip install transformer_lens fancy_einsum einops --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.6 MB/s eta 0:00:00


In [2]:
try:
  import google.colab
  IN_COLAB = True
  print("Running as a Colab notebook")
  %pip install git+https://github.com/neelnanda-io/Easy-Transformer.git@clean-transformer-demo
  # Install another version of node that makes PySvelte work way faster
  !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs
  %pip install git+https://github.com/neelnanda-io/PySvelte.git
  %pip install fancy_einsum
  %pip install einops
except:
  IN_COLAB = False
  print("Running as a Jupyter notebook - intended for development only!")

Running as a Colab notebook
  Cloning https://github.com/neelnanda-io/Easy-Transformer.git (to revision clean-transformer-demo) to /tmp/pip-req-build-rm85zzcb
  Running command git clone --filter=blob:none --quiet https://github.com/neelnanda-io/Easy-Transformer.git /tmp/pip-req-build-rm85zzcb
  Running command git checkout -b clean-transformer-demo --track origin/clean-transformer-demo
  Switched to a new branch 'clean-transformer-demo'
  Branch 'clean-transformer-demo' set up to track remote branch 'clean-transformer-demo' from 'origin'.
  Resolved https://github.com/neelnanda-io/Easy-Transformer.git to commit 1f25219e631aeb478d17075d47274db32c874e88
  Preparing metadata (setup.py) ... done
  Created wheel for easy_transformer: filename=easy_transformer-0.1.0-py3-none-any.whl size=55601 sha256=e5c25e5375d507231697effe6efd33806bde341e5faf383118340536f44dc35f
  Stored in directory: /tmp/pip-ephem-wheel-cache-mwjyhvyx/wheels/93/f3/71/f103ceb7ff1dea0b7c7d213d85708cfeb9bd35e10f18542b19
Su

In [3]:
!pip install transformer_lens

  Attempting uninstall: typeguard
    Found existing installation: typeguard 2.13.3
    Uninstalling typeguard-2.13.3:
      Successfully uninstalled typeguard-2.13.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pysvelte 1.0.0 requires typeguard~=2.0, but you have typeguard 4.5.1 which is incompatible.


In [4]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
from transformer_lens import HookedTransformer
import torch
import torch.nn as nn
import numpy as np
import math
from transformer_lens.utils import gelu_new, tokenize_and_concatenate, get_corner
import tqdm.auto as tqdm

/tmp/ipykernel_3440/1986855516.py:9: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens.utils import gelu_new, tokenize_and_concatenate, get_corner


In [5]:
reference_gpt2 = HookedTransformer.from_pretrained("gpt2-small", fold_ln=False, center_unembed=False, center_writing_weights=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


Run a reference forward pass so we have a `cache` for the tests.

In [6]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).cuda()
logits, cache = reference_gpt2.run_with_cache(tokens)

## Reference activation shapes

Key:
```
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)
```

In [7]:
for activation_name, activation in cache.cache_dict.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(activation_name, activation.shape)

hook_embed torch.Size([1, 35, 768])
hook_pos_embed torch.Size([1, 35, 768])
blocks.0.hook_resid_pre torch.Size([1, 35, 768])
blocks.0.ln1.hook_scale torch.Size([1, 35, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 35, 768])
blocks.0.attn.hook_q torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_k torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_v torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_attn_scores torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_pattern torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_z torch.Size([1, 35, 12, 64])
blocks.0.hook_attn_out torch.Size([1, 35, 768])
blocks.0.hook_resid_mid torch.Size([1, 35, 768])
blocks.0.ln2.hook_scale torch.Size([1, 35, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 35, 768])
blocks.0.mlp.hook_pre torch.Size([1, 35, 3072])
blocks.0.mlp.hook_post torch.Size([1, 35, 3072])
blocks.0.hook_mlp_out torch.Size([1, 35, 768])
blocks.0.hook_resid_post torch.Size([1, 35, 768])
ln_final.hook_scale torch.Size([1, 35, 1])
ln_final.hook_normalized torc

## Reference parameter shapes

In [8]:
for name, param in reference_gpt2.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(name, param.shape)

embed.W_E torch.Size([50257, 768])
pos_embed.W_pos torch.Size([1024, 768])
blocks.0.ln1.w torch.Size([768])
blocks.0.ln1.b torch.Size([768])
blocks.0.ln2.w torch.Size([768])
blocks.0.ln2.b torch.Size([768])
blocks.0.attn.W_Q torch.Size([12, 768, 64])
blocks.0.attn.W_O torch.Size([12, 64, 768])
blocks.0.attn.b_Q torch.Size([12, 64])
blocks.0.attn.b_O torch.Size([768])
blocks.0.attn.W_K torch.Size([12, 768, 64])
blocks.0.attn.W_V torch.Size([12, 768, 64])
blocks.0.attn.b_K torch.Size([12, 64])
blocks.0.attn.b_V torch.Size([12, 64])
blocks.0.mlp.W_in torch.Size([768, 3072])
blocks.0.mlp.b_in torch.Size([3072])
blocks.0.mlp.W_out torch.Size([3072, 768])
blocks.0.mlp.b_out torch.Size([768])
ln_final.w torch.Size([768])
ln_final.b torch.Size([768])
unembed.W_U torch.Size([768, 50257])
unembed.b_U torch.Size([50257])


## Config

In [45]:

@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 4 * 768 = 3072
d_head = 768 / 12 = 64

In [10]:
# All activation shapes of ref model
for activation_name, activation in cache.cache_dict.items():
  if ".0." in activation_name or "blocks" not in activation_name:
    print(activation_name, activation.shape)

hook_embed torch.Size([1, 35, 768])
hook_pos_embed torch.Size([1, 35, 768])
blocks.0.hook_resid_pre torch.Size([1, 35, 768])
blocks.0.ln1.hook_scale torch.Size([1, 35, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 35, 768])
blocks.0.attn.hook_q torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_k torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_v torch.Size([1, 35, 12, 64])
blocks.0.attn.hook_attn_scores torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_pattern torch.Size([1, 12, 35, 35])
blocks.0.attn.hook_z torch.Size([1, 35, 12, 64])
blocks.0.hook_attn_out torch.Size([1, 35, 768])
blocks.0.hook_resid_mid torch.Size([1, 35, 768])
blocks.0.ln2.hook_scale torch.Size([1, 35, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 35, 768])
blocks.0.mlp.hook_pre torch.Size([1, 35, 3072])
blocks.0.mlp.hook_post torch.Size([1, 35, 3072])
blocks.0.hook_mlp_out torch.Size([1, 35, 768])
blocks.0.hook_resid_post torch.Size([1, 35, 768])
ln_final.hook_scale torch.Size([1, 35, 1])
ln_final.hook_normalized torc

# Actual Implementation

In [11]:
for name, param in reference_gpt2.named_parameters():
  print(name, param.shape)


embed.W_E torch.Size([50257, 768])
pos_embed.W_pos torch.Size([1024, 768])
blocks.0.ln1.w torch.Size([768])
blocks.0.ln1.b torch.Size([768])
blocks.0.ln2.w torch.Size([768])
blocks.0.ln2.b torch.Size([768])
blocks.0.attn.W_Q torch.Size([12, 768, 64])
blocks.0.attn.W_O torch.Size([12, 64, 768])
blocks.0.attn.b_Q torch.Size([12, 64])
blocks.0.attn.b_O torch.Size([768])
blocks.0.attn.W_K torch.Size([12, 768, 64])
blocks.0.attn.W_V torch.Size([12, 768, 64])
blocks.0.attn.b_K torch.Size([12, 64])
blocks.0.attn.b_V torch.Size([12, 64])
blocks.0.mlp.W_in torch.Size([768, 3072])
blocks.0.mlp.b_in torch.Size([3072])
blocks.0.mlp.W_out torch.Size([3072, 768])
blocks.0.mlp.b_out torch.Size([768])
blocks.1.ln1.w torch.Size([768])
blocks.1.ln1.b torch.Size([768])
blocks.1.ln2.w torch.Size([768])
blocks.1.ln2.b torch.Size([768])
blocks.1.attn.W_Q torch.Size([12, 768, 64])
blocks.1.attn.W_O torch.Size([12, 64, 768])
blocks.1.attn.b_Q torch.Size([12, 64])
blocks.1.attn.b_O torch.Size([768])
blocks.1.a

## Some tests

In [27]:

# for our model (where input is floats)
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randn(shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

# for our model (where input is ints)
def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randint(100, 1000, shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

# takes instance of that layer from ref model and its original input
def load_gpt2_test(cls, gpt2_layer, input_name, cache_dict=cache.cache_dict):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    # Allow inputs of strings or tensors
    if isinstance(input_name, str):
        reference_input = cache_dict[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input)
    print("Output shape:", output.shape)

    # Attention has different set of inputs compared to other layers
    if cls.__name__ == "Attention":
      reference_output = gpt2_layer(reference_input, reference_input, reference_input)
    else:
      reference_output = gpt2_layer(reference_input)

    print("Reference output shape: ", reference_output.shape)

    comparison = torch.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct")
    return output

## LayerNorm

1.  Make mean 0
2. normalize to have variance 1
3. Scale with learned weights
4. Translate with learned bias

In [104]:
class LayerNorm(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.w = nn.Parameter(torch.ones(cfg.d_model))
    self.b = nn.Parameter(torch.zeros(cfg.d_model))

  def forward(self, residual):
    # residual: [batch, position, d_model]
    if self.cfg.debug: print("Residual:", residual.shape)
    residual = residual - einops.reduce(residual, "batch position d_model -> batch position 1", "mean") # making mean 0
    # Calculate variance, then sqrt it. Epsilon to prevent divide by 0
    scale = (einops.reduce(residual.pow(2), "batch position d_model -> batch position 1", "mean") + cfg.layer_norm_eps).sqrt()
    normalized = residual / scale
    normalized = normalized * self.w + self.b
    if self.cfg.debug: print("Normalized:", residual.shape)
    return normalized


In [29]:
# Testing layernorm
_ = rand_float_test(LayerNorm, [2, 4, 768])

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])



In [30]:
_ = load_gpt2_test(LayerNorm, reference_gpt2.ln_final, "blocks.11.hook_resid_post")

Input shape: torch.Size([1, 35, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


## Embedding
A lookup table from tokens to residual stream vectors

In [97]:
class Embed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_E = nn.Parameter(torch.empty((cfg.d_vocab, cfg.d_model)))
    nn.init.normal_(self.W_E, std = self.cfg.init_range)

  def forward(self, tokens):
    # tokens shape: [batch, positions]
    if self.cfg.debug: print("Tokens", tokens.shape)
    embed = self.W_E[tokens, :] # applying the embedding by indexing along d vocab axis, final shape = [batch, pos, d_model]
    if self.cfg.debug: print("Embeddings", embed.shape)
    return embed

In [91]:
# Testing embedding layer
rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

Input shape: torch.Size([2, 4])
Tokens torch.Size([2, 4])
Embeddings torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 40])
Tokens torch.Size([1, 40])
Embeddings torch.Size([1, 40, 768])
Output shape: torch.Size([1, 40, 768])
Reference output shape:  torch.Size([1, 40, 768])
100.00% of the values are correct


tensor([[[ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207],
         [-0.0251, -0.0796, -0.0282,  ...,  0.1481,  0.0152, -0.0240],
         [ 0.0655, -0.1402,  0.0478,  ..., -0.0409,  0.1575, -0.0507],
         ...,
         [-0.0485, -0.0868, -0.0170,  ...,  0.0239, -0.0942,  0.1051],
         [-0.1406, -0.0124,  0.0133,  ...,  0.0034, -0.0123,  0.3049],
         [ 0.0466, -0.0113,  0.0283,  ..., -0.0735,  0.0496,  0.0963]]],
       device='cuda:0', grad_fn=<IndexBackward0>)

## Positional Embedding

lookup table for positions to give each position a context instead of just each word/token like bag of words

this weight is updated on gradient descent hence the name learned absolute pos embedding

In [99]:
class PosEmbed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_pos = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model)))
    nn.init.normal_(self.W_pos, std=self.cfg.init_range)

  def forward(self, tokens):
    # tokens is [batch, position]
    if self.cfg.debug: print("Tokens:", tokens.shape)
    pos_embed = self.W_pos[:tokens.size(1), :] # [position, d_model] -> indexing by position, so taking the size of the seq
    pos_embed = einops.repeat(pos_embed, "position d_model -> batch position d_model", batch = tokens.size(0))
    if self.cfg.debug: print("pos_embed:", pos_embed.shape)
    return pos_embed


In [89]:
# Testing pos embed layer
rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

Input shape: torch.Size([2, 4])
Tokens: torch.Size([2, 4])
pos_embed: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 40])
Tokens: torch.Size([1, 40])
pos_embed: torch.Size([1, 40, 768])
Output shape: torch.Size([1, 40, 768])
Reference output shape:  torch.Size([1, 40, 768])
100.00% of the values are correct


tensor([[[-1.8821e-02, -1.9742e-01,  4.0267e-03,  ..., -4.3044e-02,
           2.8267e-02,  5.4490e-02],
         [ 2.3959e-02, -5.3792e-02, -9.4879e-02,  ...,  3.4170e-02,
           1.0172e-02, -1.5573e-04],
         [ 4.2161e-03, -8.4764e-02,  5.4515e-02,  ...,  1.9745e-02,
           1.9325e-02, -2.1424e-02],
         ...,
         [-5.2525e-04,  1.4567e-02,  4.0955e-02,  ...,  2.9406e-05,
          -2.6043e-03, -3.9064e-03],
         [ 4.0230e-03,  5.7504e-03,  3.3747e-02,  ..., -2.0234e-03,
          -2.8032e-03, -5.2562e-03],
         [ 3.0969e-03,  1.0802e-02,  3.6991e-02,  ..., -1.7777e-03,
          -5.7024e-03, -3.9348e-03]]], device='cuda:0',
       grad_fn=<ExpandBackward0>)

## Attention
1. Produce an attention pattern for each destination token - a probability dist over 0th to curr token
  * Linear map from input -> query, key, where shape: [batch, head_index, d_head]
  * then dot product every pair of queries and keys to get attention scores [batch, head_index, query_pos, key_pos] (query = dest, key = src)
  * Scale and mask attn scores to make it causal
  * softmax row-wise, to get a probability dist along each the key_pos dim -> this is the final attention pattern
2. Move info from src tokens to dest token using attention pattern (moving is via linear map)
-  Linear map from input -> value [batch, key_pos, head_index, d_head]
- Mix along the key_pos axis with attention pattern to get a mixed value [batch, query_pos, head_index, d_head]
- map to output, [batch, position, d_model] (position is query pos since we summed over all the attention heads


In [100]:
class Attention(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_Q, std=self.cfg.init_range)
    self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_K, std=self.cfg.init_range)
    self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_V, std=self.cfg.init_range)
    self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))

    self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
    nn.init.normal_(self.W_O, std=self.cfg.init_range)
    self.b_O = nn.Parameter(torch.zeros((cfg.d_model)))

    self.register_buffer("IGNORE", torch.tensor(-1e5, dtype=torch.float32, device="cuda"))
  def forward(self, normalized_resid_pre):
    # normalized_resid_pre: [batch, position, d_model]
    if self.cfg.debug: print("Normalized_resid_pre:", normalized_resid_pre.shape)

    q = einsum("batch query_pos d_model, n_heads d_model d_head -> batch query_pos n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
    k = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_K) + self.b_K

    attn_scores = einsum("batch query_pos n_heads d_head, batch key_pos n_heads d_head -> batch n_heads query_pos key_pos", q, k)
    attn_scores = attn_scores / math.sqrt(self.cfg.d_head)
    attn_scores = self.apply_causal_mask(attn_scores)

    pattern = attn_scores.softmax(dim=-1) # [batch, n_head, query_pos, key_pos]

    v = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_V) + self.b_V

    z = einsum("batch n_heads query_pos key_pos, batch key_pos n_heads d_head -> batch query_pos n_heads d_head", pattern, v)

    attn_out = einsum("batch query_pos n_heads d_head, n_heads d_head d_model -> batch query_pos d_model", z, self.W_O) + self.b_O

    if self.cfg.debug:
      print("z shape:", z.shape)
      print("W_O shape:", self.W_O.shape)

    return attn_out

  # mask to remaining tokens in a sequence to maintain backward looking property properly
  def apply_causal_mask(self, attn_scores):
    mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device), diagonal = 1).bool()
    attn_scores.masked_fill_(mask, self.IGNORE)
    return attn_scores



In [35]:
# Testing attention layer
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["blocks.0.ln1.hook_normalized"])

Input shape: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Normalized_resid_pre: torch.Size([1, 35, 768])
z shape: torch.Size([1, 35, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[ 1.3649e+00,  2.1711e+00,  7.0824e+00,  ..., -1.4679e-01,
           2.6480e-01,  9.8746e-01],
         [-1.3159e+01, -4.1196e+00,  8.6870e+00,  ..., -4.7698e-01,
          -2.4685e-01,  3.7986e-01],
         [-1.7002e+01,  4.8321e+00, -6.2118e-01,  ..., -7.1945e-01,
           1.0781e+00,  5.4464e-01],
         ...,
         [-1.3211e+01,  7.5173e-01,  8.9662e+00,  ..., -4.2861e-01,
           4.6559e-01, -9.4983e-01],
         [-1.3985e-03,  6.5740e+00,  1.9785e+01,  ..., -6.7092e-01,
          -1.0935e-01,  7.8003e-02],
         [-6.0138e+00, -1.8512e-01,  1.8866e+01,  ..., -5.4550e-01,
          -4.9668e-02, -1.4721e-01]]], device='cuda:0', grad_fn=<AddBackward0>)

## MLP

In [101]:
class MLP(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_in = nn.Parameter(torch.empty((cfg.d_model, cfg.d_mlp)))
    nn.init.normal_(self.W_in, std=self.cfg.init_range)
    self.b_in = nn.Parameter(torch.zeros(cfg.d_mlp))

    self.W_out = nn.Parameter(torch.empty((cfg.d_mlp, cfg.d_model)))
    nn.init.normal_(self.W_out, std=self.cfg.init_range)
    self.b_out = nn.Parameter(torch.zeros(cfg.d_model))

  def forward(self, normalized_resid_mid):
    # normalized resid mid stream: [batch, position, d_model]
    if self.cfg.debug: print("Normalized residual stream mid:", normalized_resid_mid.shape)

    pre = einsum("batch position d_model, d_model d_mlp -> batch position d_mlp", normalized_resid_mid, self.W_in) + self.b_in
    post = gelu_new(pre)
    mlp_out = einsum("batch position d_mlp, d_mlp d_model -> batch position d_model", post, self.W_out) + self.b_out
    return mlp_out






In [39]:
# Testing the mlp layer

rand_float_test(MLP, [2, 4, 768])
load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["blocks.0.ln1.hook_normalized"])

Input shape: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Normalized residual stream mid: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[  0.0274,   2.4044,   2.6138,  ...,  13.3645,   8.4272,  -0.8360],
         [ -8.6545,   1.7920,   0.6268,  ...,  -5.1794,  -1.2061,   5.3991],
         [-10.3536, -13.2765,  -6.0768,  ...,   4.8616,  -1.8622,  10.4763],
         ...,
         [ -8.7080,   7.6277,   6.4608,  ...,   4.3486,  -5.2200,   9.6780],
         [-11.1025,   2.4969,  21.4818,  ...,  -0.5803,  -2.8098,  12.2156],
         [  3.2533,  -0.9791,  16.5849,  ...,   6.3753,   8.0988,   8.1493]]],
       device='cuda:0', grad_fn=<AddBackward0>)

## Transformer Block

In [54]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg

    self.ln1 = LayerNorm(cfg)
    self.attn = Attention(cfg)
    self.ln2 = LayerNorm(cfg)
    self.mlp = MLP(cfg)

  def forward(self, resid_pre):
    # Resid pre is shape [batch, position, d_model]

    # Attention layer and updating residual stream
    normalized_resid_pre = self.ln1(resid_pre)
    attn_out = self.attn(normalized_resid_pre)
    resid_mid = resid_pre + attn_out

    # MLP layer and updating final resid stream output
    normalized_resid_mid = self.ln2(resid_mid)
    mlp_out = self.mlp(normalized_resid_mid)
    resid_out = resid_mid + mlp_out

    return resid_out

In [55]:
# Testing one transformer block

rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])

Input shape: torch.Size([1, 35, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Normalized_resid_pre: torch.Size([1, 35, 768])
z shape: torch.Size([1, 35, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([1, 35, 768])
Normalized: torch.Size([1, 35, 768])
Normalized residual stream mid: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape:  torch.Size([1, 35, 768])
100.00% of the values are correct


tensor([[[ 0.3911,  0.1543,  0.6005,  ...,  1.7198,  1.7365,  0.3930],
         [-0.9039, -0.0360,  0.2351,  ..., -0.4148,  0.3562,  0.3936],
         [-0.9647, -2.4819, -1.4995,  ...,  1.4046,  0.7616,  0.5918],
         ...,
         [-0.7421,  0.9251, -0.3218,  ...,  0.2921,  0.1097, -0.5344],
         [-1.3221,  0.8960,  1.1795,  ..., -0.5544, -0.4071,  0.9255],
         [ 1.1209, -0.8919,  1.3737,  ..., -0.1356,  0.3434,  0.4517]]],
       device='cuda:0', grad_fn=<AddBackward0>)

## Unembedding

In [56]:
class Unembed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_U = nn.Parameter(torch.empty((cfg.d_model, cfg.d_vocab)))
    nn.init.normal_(self.W_U, std=self.cfg.init_range)
    self.b_U = nn.Parameter(torch.zeros((cfg.d_vocab)))

  def forward(self, normalized_resid_final):
    # normalized resid final/out = [batch, position, d_model]
    if self.cfg.debug: print("Normalized_resid_final:", normalized_resid_final.shape)
    logits = einsum("batch position d_model, d_model d_vocab -> batch position d_vocab", normalized_resid_final, self.W_U) + self.b_U
    return logits


## Full Transformer

In [102]:
class DemoTransformer(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.embed = Embed(cfg)
    self.pos_embed = PosEmbed(cfg)
    self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
    self.ln_final = LayerNorm(cfg)
    self.unembed = Unembed(cfg)

  '''
  def forward(self, tokens):
    # tokens [batch, position]
    embed = self.embed(tokens)
    pos_embed = self.pos_embed(tokens)
    resid = embed + pos_embed
    for block in self.blocks:
      resid = block(resid)
    normalized_resid_final = self.ln_final(resid)
    logits = self.unembed(normalized_resid_final)
    # final shape: [batch, position, logits]
    return logits
'''
  # forwarding logic but saving mid outputs to a cache
  def forward(self, tokens, return_cache=False):
    # tokens [batch, position]
    cache = {}
    embed = self.embed(tokens)
    pos_embed = self.pos_embed(tokens)
    resid = embed + pos_embed
    cache["resid_initial"] = resid

    for i, block in enumerate(self.blocks):
        normalized_resid_pre = block.ln1(resid)
        attn_out = block.attn(normalized_resid_pre)
        resid_mid = resid + attn_out
        cache[f"block_{i}.resid_mid"] = resid_mid

        normalized_resid_mid = block.ln2(resid_mid)
        mlp_out = block.mlp(normalized_resid_mid)
        resid = resid_mid + mlp_out
        cache[f"block_{i}.resid_post"] = resid

  # final shape: [batch, position, logits]
    normalized_resid_final = self.ln_final(resid)
    logits = self.unembed(normalized_resid_final)
    return (logits, cache) if return_cache else logits


In [106]:
rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

Input shape: torch.Size([2, 4])
Tokens torch.Size([2, 4])
Embeddings torch.Size([2, 4, 768])
Tokens: torch.Size([2, 4])
pos_embed: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized residual stream mid: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Normalized_resid_pre: torch.Size([2, 4, 768])
z shape: torch.Size([2, 4, 12, 64])
W_O shape: torch.Size([12, 64, 768])
Residual: torch.Size([2, 4

tensor([[[ -43.4317,  -39.8364,  -43.0660,  ...,  -54.0877,  -54.3452,
           -42.3644],
         [ -43.5870,  -43.6667,  -47.2854,  ...,  -51.6409,  -51.4358,
           -44.0636],
         [ -84.3084,  -84.6175,  -85.7358,  ...,  -96.5972,  -93.6002,
           -87.1822],
         ...,
         [ -92.9680,  -92.9539,  -97.1435,  ..., -100.6460,  -99.9687,
           -93.6020],
         [ -88.0357,  -90.3075,  -95.9869,  ...,  -97.6964,  -99.2046,
           -90.6128],
         [-132.1418, -132.4836, -133.4077,  ..., -140.0103, -141.6410,
          -125.1865]]], device='cuda:0', grad_fn=<AddBackward0>)

# Trying out the model

In [107]:
demo_gpt2 = DemoTransformer(Config(debug=False))
demo_gpt2.cfg.debug = False
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)
demo_gpt2.cuda()

DemoTransformer(
  (embed): Embed()
  (pos_embed): PosEmbed()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm()
      (attn): Attention()
      (ln2): LayerNorm()
      (mlp): MLP()
    )
  )
  (ln_final): LayerNorm()
  (unembed): Unembed()
)

In [116]:
test_thing = """In just a few short years, rapper Playboi Carti has amassed over 19B streams worldwide. Carti has been unstoppable since the release of his 2017 single "Magnolia," as its meteoric rise garnered cosigns from Beyoncé and features on series like Atlanta. His self-titled album has accumulated nearly 7.8B streams to date after debuting at #12 on the Billboard 200 chart where it spent 63 weeks. The following year, Carti dropped his album Die Lit, debuting at #3 on the Billboard 200, with collaborations like Lil Uzi Vert on "Shoota,"
"Poke It Out" with Nicki Minaj & "Love Hurts" with Travis Scott. The album has nearly 9.5B global streams to date and spent a total of 11 weeks on the Billboard 200. In April of 2020, Playboi Carti returned with track "@MEH," and on Christmas day, he landed his first #1 album on Billboard's 200 Chart with Whole Lotta Red which has amassed a staggering 9.4B global streams to date. Playboi Carti kicked off 2024 strong with new music & collaborations including
"CARNIVAL" with Kanye West, Ty Dolla $ign, and Rich the Kid, reaching #1 on the Billboard Hot 100 chart. Carti's collaboration with Travis Scott on their song "FE!N" peaked at #5 on Billboard's Hot 100. Playboi Carti has also recently featured on "I LUV IT" with Camila Cabello, "TYPE SHIT" with Future, Metro Boomin, and Travis Scott, and "Popular" with The Weeknd and Madonna. His most recent collaboration "Timeless" with The Weeknd debuted at #3 ol Billboard Hot 100."""

In [108]:
tokens_demo = reference_gpt2.to_tokens(test_thing).cuda()
demo_logits = demo_gpt2(tokens_demo)

Normalized_resid_final: torch.Size([1, 367, 768])


In [109]:
# seeing top prediction for every position

import torch.nn.functional as F

for i in range(cfg.n_layers):
    resid = cache[f"block_{i}.resid_post"]
    normalized = demo_gpt2.ln_final(resid)
    block_logits = demo_gpt2.unembed(normalized)  # [1, 367, d_vocab]

    # Top token at every position
    top_tokens = block_logits[0].argmax(dim=-1)  # [367]
    decoded = reference_gpt2.to_string(top_tokens)

    print(f"=== Block {i} ===")
    print(decoded)
    print()

Normalized_resid_final: torch.Size([1, 367, 768])
=== Block 0 ===

 addition one few hundred short ago which rapper Playboye Cartye been amasseddrivethBS streams worldwide
 Cartye been seen unstoppable since same release the own 2017 singlenoificent Trees while well own meteoric rise garnered cosignalsa the Beyoncé then features top series ours Atlanta And own selfbasedteitled album been accumulated nearlyth But88rick streams be date the debut the least #31 the same Billboard times chartabouts's spentrd ago
 most follows ago which Cartwa dropped own album Die Lit which debuting least #rd the same Billboard times which the collaborations a LilUzi Vert thenoShoot few while
Auppetokeself OutA the Nickwa Minaj/no Love Hurts/" the Travis Scott
 latter album been nearly09 Thetherk warming streams be date then spent few number theth ago the same Billboard times
 addition 2015 the 2020 although Playbopec Cartpec return the track "@MEOLD according then the Christmas day although'll landed own a

## Evaluating model on sample text above
This is pretty similar to "news" type of text that gpt2 was originally trained on. However, the actual subject matter (rapper playboi carti, who gained this level of fame after 2019) is unfamiliar to the model since gpt2 was trained on text up till december 2017. Still, we can expect a low loss since majority of the content in the sample text reads similiar to the patterns that were availible in gpt2's training.

In [110]:
def lm_cross_entropy_loss(logits, tokens):
  # measure next token loss
  # logits having shape batch, pos, d_vocab
  # tokens have shape batch, position
  log_probs = logits.log_softmax(dim=-1)
  pred_log_probs = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)
  return -pred_log_probs.mean()

loss = lm_cross_entropy_loss(demo_logits, tokens_demo)
print("Raw loss:", loss)
print("Loss as average prob: ", (-loss).exp())
print("Uniform loss over the vocab", math.log(demo_gpt2.cfg.d_vocab))

Raw loss: tensor(2.9861, device='cuda:0', grad_fn=<NegBackward0>)
Loss as average prob:  tensor(0.0505, device='cuda:0', grad_fn=<ExpBackward0>)
Uniform loss over the vocab 10.82490511970208


## Generating text
num_chars = chars to generate

In [128]:
#test_thing = """Breaking News: President Trump has been impeached by the House of Representatives for abuse of power and obstruction of Congress. The vote was 230 to 197, with 10 Republicans joining all Democrats in voting to The House will now send the articles of impeachment to the Senate, where a trial will be held to determine whether to remove the president from"""
test_thing = """In just a few short years, rapper Playboi Carti has amassed over 19B streams worldwide. Carti has been unstoppable since the release of his 2017 single "Magnolia," as its meteoric rise garnered cosigns from Beyoncé and features on series like Atlanta. His self-titled album has accumulated nearly 7.8B streams to date after debuting at #12 on the Billboard 200 chart where it spent 63 weeks. The following year, Carti dropped his album Die Lit, debuting at #3 on the Billboard 200, with collaborations like Lil Uzi Vert on "Shoota,"
"Poke It Out" with Nicki Minaj & "Love Hurts" with Travis Scott. The album has nearly 9.5B global streams to date and spent a total of 11 weeks on the Billboard 200. In April of 2020, Playboi Carti returned with track "@MEH," and on Christmas day, he landed his first #1 album on Billboard's 200 Chart with Whole Lotta Red which has amassed a staggering 9.4B global streams to date. Playboi Carti kicked off 2024 strong with new music & collaborations including
"CARNIVAL" with Kanye West, Ty Dolla $ign, and Rich the Kid, reaching #1 on the Billboard Hot 100 chart. Carti's collaboration with Travis Scott on their song "FE!N" peaked at #5 on Billboard's Hot 100. Playboi Carti has also recently featured on "I LUV IT" with Camila Cabello, "TYPE SHIT" with Future, Metro Boomin, and Travis Scott, and "Popular" with The Weeknd and Madonna. His most recent collaboration "Timeless" with The Weeknd debuted at #3 ol Billboard Hot 100."""

num_chars = 60
for i in tqdm.tqdm(range(num_chars)):
    test_tokens = reference_gpt2.to_tokens(test_thing).cuda()
    demo_logits = demo_gpt2(test_tokens)   # use test_tokens, not tokens_demo
    next_token_id = demo_logits[0, -1].argmax()   # also: [0, -1] not [-1, -1]
    test_thing += reference_gpt2.tokenizer.decode(next_token_id)



  0%|          | 0/60 [00:00<?, ?it/s]

Normalized_resid_final: torch.Size([1, 367, 768])
Normalized_resid_final: torch.Size([1, 368, 768])
Normalized_resid_final: torch.Size([1, 368, 768])
Normalized_resid_final: torch.Size([1, 369, 768])
Normalized_resid_final: torch.Size([1, 370, 768])
Normalized_resid_final: torch.Size([1, 371, 768])
Normalized_resid_final: torch.Size([1, 372, 768])
Normalized_resid_final: torch.Size([1, 373, 768])
Normalized_resid_final: torch.Size([1, 374, 768])
Normalized_resid_final: torch.Size([1, 375, 768])
Normalized_resid_final: torch.Size([1, 376, 768])
Normalized_resid_final: torch.Size([1, 377, 768])
Normalized_resid_final: torch.Size([1, 378, 768])
Normalized_resid_final: torch.Size([1, 379, 768])
Normalized_resid_final: torch.Size([1, 380, 768])
Normalized_resid_final: torch.Size([1, 381, 768])
Normalized_resid_final: torch.Size([1, 382, 768])
Normalized_resid_final: torch.Size([1, 383, 768])
Normalized_resid_final: torch.Size([1, 384, 768])
Normalized_resid_final: torch.Size([1, 385, 768])


In [129]:
print(test_thing)

In just a few short years, rapper Playboi Carti has amassed over 19B streams worldwide. Carti has been unstoppable since the release of his 2017 single "Magnolia," as its meteoric rise garnered cosigns from Beyoncé and features on series like Atlanta. His self-titled album has accumulated nearly 7.8B streams to date after debuting at #12 on the Billboard 200 chart where it spent 63 weeks. The following year, Carti dropped his album Die Lit, debuting at #3 on the Billboard 200, with collaborations like Lil Uzi Vert on "Shoota,"
"Poke It Out" with Nicki Minaj & "Love Hurts" with Travis Scott. The album has nearly 9.5B global streams to date and spent a total of 11 weeks on the Billboard 200. In April of 2020, Playboi Carti returned with track "@MEH," and on Christmas day, he landed his first #1 album on Billboard's 200 Chart with Whole Lotta Red which has amassed a staggering 9.4B global streams to date. Playboi Carti kicked off 2024 strong with new music & collaborations including
"CARN

## Other Texts

The texts are organized from most commonly found patterns in GPT2's original training text to things that aren't found at all. So, loss should go up as lower strings are tested

In [111]:
test_texts = [
    # should be low loss
    "The early bird catches the worm. A penny saved is a penny earned. Better late than never.",


    "The president addressed the nation on Tuesday, announcing new economic policies aimed at reducing inflation and supporting working families across the country.",


    "Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics.",

    # now medium loss
    "So I was walking down the street yesterday and I ran into my old friend from high school. We hadn't seen each other in years.",


    "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!",

    # older english so slightly higher loss
    "It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness.",

    # code, which the model is not trained for
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",

    # technical jargon
    "The mechanistic interpretability of transformer-based language models reveals emergent circuits performing in-context learning through induction heads.",

    # no patterns here - should be high loss
    "Purple elephant banana quantum sandwich telescope yesterday running purple. Banana telescope elephant quantum.",

    # not english - should be very high loss
    "El gato negro corrió rápidamente a través del jardín mientras los pájaros cantaban en los árboles altos.",
]

print(f"{'#':<3} {'Loss':<8} {'Avg Prob':<10} {'Perplexity':<12} Description")
print("-" * 80)

for i, text in enumerate(test_texts, 1):
    tokens = reference_gpt2.to_tokens(text).cuda()
    logits = demo_gpt2(tokens)
    loss = lm_cross_entropy_loss(logits, tokens)
    avg_prob = (-loss).exp()
    perplexity = loss.exp()
    desc = text[:50].replace('\n', ' ') + "..."
    print(f"{i:<3} {loss.item():<8.3f} {avg_prob.item():<10.4f} {perplexity.item():<12.2f} {desc}")

#   Loss     Avg Prob   Perplexity   Description
--------------------------------------------------------------------------------
Normalized_resid_final: torch.Size([1, 21, 768])
1   3.715    0.0244     41.06        The early bird catches the worm. A penny saved is ...
Normalized_resid_final: torch.Size([1, 25, 768])
2   2.876    0.0564     17.74        The president addressed the nation on Tuesday, ann...
Normalized_resid_final: torch.Size([1, 26, 768])
3   2.371    0.0934     10.71        Albert Einstein was a German-born theoretical phys...
Normalized_resid_final: torch.Size([1, 29, 768])
4   2.237    0.1068     9.37         So I was walking down the street yesterday and I r...
Normalized_resid_final: torch.Size([1, 35, 768])
5   4.565    0.0104     96.04        I am an amazing autoregressive, decoder-only, GPT-...
Normalized_resid_final: torch.Size([1, 30, 768])
6   1.872    0.1538     6.50         It was the best of times, it was the worst of time...
Normalized_resid_final: torch.

# Training a model

In [149]:
# config
batch_size = 8
num_epochs = 1
log_every = 10
lr = 1e-3
weight_decay = 1e-2
model_cfg = Config(debug=False, d_model=256, n_heads=4, d_head=64, d_mlp=1024, n_layers=2, n_ctx=1024, d_vocab=reference_gpt2.cfg.d_vocab)

Create data

In [156]:
from datasets import load_dataset

dataset = load_dataset("NeelNanda/pile-10k", split="train")
print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'meta'],
    num_rows: 10000
})
{'text': 'It is done, and submitted. You can play “Survival of the Tastiest” on Android, and on the web. Playing on the web works, but you have to simulate multi-touch for table moving and that can be a bit confusing.\n\nThere’s a lot I’d like to talk about. I’ll go through every topic, insted of making the typical what went right/wrong list.\n\nConcept\n\nWorking over the theme was probably one of the hardest tasks I had to face.\n\nOriginally, I had an idea of what kind of game I wanted to develop, gameplay wise – something with lots of enemies/actors, simple graphics, maybe set in space, controlled from a top-down view. I was confident I could fit any theme around it.\n\nIn the end, the problem with a theme like “Evolution” in a game is that evolution is unassisted. It happens through several seemingly random mutations over time, with the most apt permutation surviving. This genetic car simulator is, in my opinion, a g

In [162]:

tokens_dataset = tokenize_and_concatenate(dataset, reference_gpt2.tokenizer, streaming=False, max_length=model_cfg.n_ctx, column_name="text", add_bos_token=True, num_proc=4)
data_loader = torch.utils.data.DataLoader(tokens_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

Create model

In [158]:
model = DemoTransformer(model_cfg)
model.cuda()


DemoTransformer(
  (embed): Embed()
  (pos_embed): PosEmbed()
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm()
      (attn): Attention()
      (ln2): LayerNorm()
      (mlp): MLP()
    )
  )
  (ln_final): LayerNorm()
  (unembed): Unembed()
)

Create optimizer

In [159]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

Run training loop

In [166]:
losses = []
print("Number of batches: ", len(data_loader))

for epoch in range(num_epochs):
  for c, batch in tqdm.tqdm(enumerate(data_loader)):
    tokens = batch['tokens'].cuda()
    logits = model(tokens)
    loss = lm_cross_entropy_loss(logits, tokens)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss.item())
    if c % log_every:
      print(f"Step: {c}, Loss: {loss.item():.4f}")
      losses.append(loss.item())

Number of batches:  2121


0it [00:00, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Normalized_resid_final: torch.Size([8, 1024, 256])
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 1, Loss: 5.6711
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 2, Loss: 6.4342
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 3, Loss: 5.2484
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 4, Loss: 4.6146
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 5, Loss: 5.5078
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 6, Loss: 5.5150
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 7, Loss: 4.7077
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 8, Loss: 5.7526
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 9, Loss: 5.5995
Normalized_resid_final: torch.Size([8, 1024, 256])
Normalized_resid_final: torch.Size([8, 1024, 256])


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^

...

Step: 34, Loss: 4.6764
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 35, Loss: 5.5689
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 36, Loss: 5.3138
Normalized_resid_final: torch.Size([8, 1024, 256])


^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr

Step: 37, Loss: 5.4289
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 38, Loss: 5.1396
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 39, Loss: 5.0842
Normalized_resid_final: torch.Size([8, 1024, 256])
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 41, Loss: 5.3236
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 42, Loss: 4.7703
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 43, Loss: 5.1634
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 44, Loss: 4.1954
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 45, Loss: 4.5264
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 46, Loss: 5.7707
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 47, Loss: 5.3965
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 48, Loss: 4.9345
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 49, Loss: 6.2924
Normalized_resid_final: torch.Size([8, 1024, 256])
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 51, 

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e33ac03c4a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Step: 235, Loss: 6.0039
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 236, Loss: 6.1804
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 237, Loss: 4.4710
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 238, Loss: 6.0212
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 239, Loss: 4.6226
Normalized_resid_final: torch.Size([8, 1024, 256])
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 241, Loss: 5.6922
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 242, Loss: 5.6037
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 243, Loss: 5.6497
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 244, Loss: 5.4568
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 245, Loss: 5.8746
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 246, Loss: 5.5443
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 247, Loss: 5.2024
Normalized_resid_final: torch.Size([8, 1024, 256])
Step: 248, Loss: 5.2294
Normalized_resid_final: t

loss went down by a small bit - result of just one epoch

In [ ]:

from google.colab import _message
import json
nb = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
nb.get("metadata", {}).pop("widgets", None)
with open("gpt2_clean.ipynb", "w") as f:
    json.dump(nb, f, indent=1)
from google.colab import files
files.download("gpt2_clean.ipynb")